<a href="https://colab.research.google.com/github/jadenfix/masters_thesis_cold_runs/blob/main/combined_data_fetcher_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile .env
# --- .env file ---

ETHERSCAN_API_KEY="put it here"

Writing .env


In [3]:
# Required libraries: pandas, requests, python-dotenv, feedparser, newspaper3k, twikit, nest_asyncio, lxml, lxml_html_clean
# Install in Colab using:
!pip install pandas requests python-dotenv feedparser newspaper3k twikit nest_asyncio "lxml[html_clean]" -q

import os
import asyncio
import pandas as pd
import requests
import feedparser
import time
from datetime import datetime, timedelta, timezone
from newspaper import Article # For parsing news article content
from twikit import Client
from twikit.errors import TooManyRequests # Import specific error
from dotenv import load_dotenv
import nest_asyncio
import traceback # Import traceback for better error logging

# Apply nest_asyncio to allow running asyncio functions synchronously if needed
# This is often required in scripts or environments like Jupyter/Colab
nest_asyncio.apply()

# Load environment variables from .env file
# Make sure the .env file exists in the Colab environment's root directory
# or provide the path: load_dotenv(dotenv_path='/path/to/your/.env')
load_dotenv()

# --- Configuration ---
COINBASE_API_URL = "https://api.exchange.coinbase.com"
NEWS_RSS_URL_TEMPLATE = "https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"

# --- Load Credentials ---
# Use the standard variable names defined in your .env file
TWITTER_USERNAME = os.getenv("TWITTER_USERNAME")
TWITTER_EMAIL = os.getenv("TWITTER_EMAIL")
TWITTER_PASSWORD = os.getenv("TWITTER_PASSWORD")
# --- END OF CREDENTIAL LOADING ---


# --- Helper Functions ---

def standardize_timestamp(ts):
    """Converts various timestamp formats to timezone-aware UTC pandas Timestamps."""
    if isinstance(ts, pd.Timestamp):
        if ts.tzinfo is None:
            return ts.tz_localize('UTC')
        return ts.tz_convert('UTC')
    elif isinstance(ts, datetime):
        if ts.tzinfo is None:
            return pd.Timestamp(ts, tz='UTC')
        return pd.Timestamp(ts).tz_convert('UTC')
    elif isinstance(ts, (int, float)): # Assume POSIX timestamp
        try:
            # Handle potential large integer timestamps (e.g., milliseconds)
            if ts > 2**32: # A rough check if it might be milliseconds
                 ts = ts / 1000
            # Ensure timestamp is within reasonable bounds for fromtimestamp
            # Check if reasonable (e.g., between year 1971 and 10 years in the future)
            if ts < 31536000 or ts > datetime.now(timezone.utc).timestamp() + (10 * 365 * 86400):
                 raise ValueError("Timestamp out of reasonable range")
            return pd.Timestamp.utcfromtimestamp(ts)
        except (ValueError, OSError) as e: # Catch OSError too for out of range
             print(f"Warning: Could not convert numeric timestamp '{ts}': {e}. Returning NaT.")
             return pd.NaT
    elif isinstance(ts, str):
        try:
            # Try parsing standard formats, explicitly handle timezone if present
            dt = pd.to_datetime(ts, errors='coerce', utc=True) # Try parsing directly as UTC
            if pd.isna(dt):
                 # Fallback for formats without explicit timezone offset
                 dt = pd.to_datetime(ts, errors='coerce')
                 if pd.isna(dt):
                      raise ValueError("Pandas could not parse the string")
                 # Localize to UTC if naive
                 if dt.tzinfo is None:
                      # Be careful assuming UTC for naive strings, but it's a common case
                      dt = dt.tz_localize('UTC')

            return dt.tz_convert('UTC') # Ensure final is UTC
        except Exception as e:
            print(f"Warning: Could not parse timestamp string '{ts}': {e}. Returning NaT.")
            return pd.NaT # Not a Time
    else:
        print(f"Warning: Unhandled timestamp type '{type(ts)}'. Returning NaT.")
        return pd.NaT


# --- Price Data Fetcher ---

def fetch_historical_prices(product_id, days_history=365, granularity=86400):
    """
    Fetch historical candlestick data for a cryptocurrency pair from Coinbase API.

    Args:
        product_id (str): The product ID (e.g., 'BTC-USD').
        days_history (int): Number of days of historical data to fetch.
        granularity (int): Desired time slice in seconds (e.g., 86400 for daily).

    Returns:
        pd.DataFrame: DataFrame with columns ['timestamp', 'coin', 'source', 'low', 'high', 'open', 'close', 'volume']
                      Returns empty DataFrame on failure.
    """
    print(f"Fetching historical prices for {product_id}...")
    if not product_id.endswith('-USD'):
        product_id += '-USD'
    product_id = product_id.upper()
    coin = product_id.split('-')[0]
    url = f"{COINBASE_API_URL}/products/{product_id}/candles"

    end_dt = datetime.now(timezone.utc)
    start_dt = end_dt - timedelta(days=days_history)

    all_data = []
    current_start = start_dt

    # Coinbase API has a limit of 300 data points per request
    max_candles_per_request = 300
    delta_per_request = timedelta(seconds=granularity * max_candles_per_request)

    while current_start < end_dt:
        # Ensure end time doesn't exceed the current time
        current_end = min(current_start + delta_per_request, end_dt)

        # Format timestamps for the API request (ISO 8601)
        start_iso = current_start.isoformat(timespec='seconds')
        end_iso = current_end.isoformat(timespec='seconds')

        # Remove timezone offset for Coinbase API if it causes issues (API might assume UTC)
        start_iso = start_iso.split('+')[0]
        end_iso = end_iso.split('+')[0]


        params = {
            'start': start_iso,
            'end': end_iso,
            'granularity': granularity
        }
        print(f"  Fetching data from {start_iso} to {end_iso}")

        try:
            response = requests.get(url, params=params, timeout=20) # Increased timeout
            response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
            data = response.json()

            # Handle potential empty list or non-list response
            if isinstance(data, list) and data:
                 # Sort data from API (Coinbase returns oldest first)
                 data.sort(key=lambda x: x[0])
                 all_data.extend(data)
                 # Update current_start for the next iteration based on the last timestamp received
                 last_timestamp_in_batch = data[-1][0]
                 current_start = datetime.fromtimestamp(last_timestamp_in_batch, timezone.utc) + timedelta(seconds=granularity)

            elif not data:
                 print("  Received empty data list, might be end of history or API issue.")
                 # Small step forward to avoid infinite loop if end_dt is far ahead
                 current_start += delta_per_request
                 # break # Option to break if empty list means end of data
            else:
                 print(f"  Received unexpected data format: {type(data)}. Stopping.")
                 break

            # Wait briefly to avoid hitting rate limits
            time.sleep(0.5) # Adjust sleep time as needed

        except requests.exceptions.HTTPError as http_err:
             print(f"  HTTP error occurred: {http_err} - Status Code: {response.status_code}")
             if response.status_code == 429: # Too Many Requests
                 print("  Rate limit likely hit. Waiting longer...")
                 time.sleep(5) # Wait longer for rate limit
             else:
                 break # Break on other HTTP errors
        except requests.exceptions.RequestException as e:
            print(f"  Error fetching price data chunk: {e}")
            break # Break on other request errors for simplicity
        except Exception as e:
             print(f"  An unexpected error occurred processing price data: {e}")
             traceback.print_exc() # Print full traceback for unexpected errors
             break


    if not all_data:
        print(f"  No price data fetched for {product_id}.")
        return pd.DataFrame()

    # Process fetched data
    columns = ['raw_timestamp', 'low', 'high', 'open', 'close', 'volume']
    df = pd.DataFrame(all_data, columns=columns)

    # Standardize timestamp using the helper function
    df['timestamp'] = df['raw_timestamp'].apply(standardize_timestamp)
    df = df.drop(columns=['raw_timestamp'])
    df = df.dropna(subset=['timestamp']) # Remove rows where timestamp parsing failed

    df['coin'] = coin
    df['source'] = 'Coinbase'

    # Ensure correct data types
    for col in ['low', 'high', 'open', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Sort by time and remove duplicates just in case
    df = df.sort_values('timestamp').drop_duplicates(subset=['timestamp'], keep='first')

    print(f"  Successfully fetched {len(df)} price data points for {product_id}.")
    return df[['timestamp', 'coin', 'source', 'low', 'high', 'open', 'close', 'volume']]


# --- News Data Fetcher ---

def fetch_google_news_rss(query, max_articles=50):
    """
    Fetches news articles from Google News RSS for a given query.

    Args:
        query (str): The search query (e.g., "Bitcoin").
        max_articles (int): Maximum number of articles to return.

    Returns:
        pd.DataFrame: DataFrame with columns ['timestamp', 'coin', 'source', 'title', 'url', 'content']
                      Returns empty DataFrame on failure.
    """
    print(f"Fetching Google News RSS for query: '{query}'...")
    # Use requests.utils.quote for proper URL encoding
    rss_url = NEWS_RSS_URL_TEMPLATE.format(query=requests.utils.quote(query))
    news_items = []
    # Assume first word is the coin for tagging, handle potential multi-word coins better if needed
    coin_name = query.split()[0]

    try:
        # Use a timeout for the request
        headers = {'User-Agent': 'Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)'} # Use a common bot user agent
        response = requests.get(rss_url, timeout=15, headers=headers)
        response.raise_for_status() # Check for HTTP errors
        # Parse the feed content
        feed = feedparser.parse(response.content)
    except requests.exceptions.RequestException as e:
        print(f"  Error fetching RSS feed for '{query}': {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"  Error parsing RSS feed for '{query}': {e}")
        traceback.print_exc()
        return pd.DataFrame()

    if not feed.entries:
        print(f"  No news entries found for '{query}'.")
        return pd.DataFrame()

    print(f"  Found {len(feed.entries)} potential news items.")
    count = 0
    for entry in feed.entries:
        if count >= max_articles:
            break

        title = entry.get("title", "N/A")
        url = entry.get("link", "N/A")
        published_str = entry.get("published", None) # Get the raw published string

        # Attempt to parse publication date using the standardized function
        timestamp = standardize_timestamp(published_str)
        if pd.isna(timestamp):
            # Fallback to current time if parsing fails
            timestamp = standardize_timestamp(datetime.now(timezone.utc))

        # Content fetching is disabled by default - uncomment block below to enable
        content = "Content fetching disabled"

        # --- UNCOMMENT BELOW TO ENABLE CONTENT FETCHING ---
        # --- Be mindful of potential errors, rate limits, and script execution time ---
        # if url != "N/A" and url.startswith('http'): # Basic URL validation
        #     try:
        #         print(f"    Fetching content for: {title[:50]}...")
        #         # Use requests to download HTML, then pass to Article
        #         html_content = requests.get(url, headers=headers, timeout=10).text
        #         article = Article(url)
        #         article.download(input_html=html_content) # Pass downloaded HTML
        #         article.parse()
        #         content = article.text
        #         time.sleep(0.5) # Be respectful to the source websites
        #     except Exception as e:
        #         print(f"    Warning: Could not fetch content for {url}: {e}")
        #         content = f"Error fetching content: {e}"
        # else:
        #      content = "Invalid or missing URL"
        # --- END OF CONTENT FETCHING BLOCK ---

        news_items.append({
            "timestamp": timestamp,
            "coin": coin_name, # Tag with the primary query term
            "source": "GoogleNews",
            "title": title,
            "url": url,
            "content": content # Store the fetched content or placeholder/error
        })
        count += 1

    if not news_items:
        return pd.DataFrame()

    df = pd.DataFrame(news_items)
    # Drop rows where timestamp is NaT after all attempts
    df = df.dropna(subset=['timestamp'])
    print(f"  Successfully processed {len(df)} news items for '{query}'.")
    return df[['timestamp', 'coin', 'source', 'title', 'url', 'content']]


# --- Twitter Data Fetcher ---

async def fetch_twitter_data_async(query, max_tweets=100, max_retries=3, initial_delay=5):
    """
    Asynchronously fetches recent tweets matching a query using Twikit.

    Args:
        query (str): The search query (e.g., "Bitcoin lang:en").
        max_tweets (int): Maximum number of tweets to attempt fetching.
        max_retries (int): Maximum number of retries on rate limit errors.
        initial_delay (int): Initial delay in seconds before retrying.

    Returns:
        pd.DataFrame: DataFrame with columns ['timestamp', 'coin', 'source', 'tweet_id', 'user', 'tweet_text']
                      Returns empty DataFrame on failure.
    """
    print(f"Fetching Twitter data for query: '{query}'...")
    # Check if credentials exist and are not None or empty strings
    if not TWITTER_USERNAME or not TWITTER_EMAIL or not TWITTER_PASSWORD:
        print("  Twitter credentials missing or empty in environment variables. Skipping.")
        return pd.DataFrame()
    else:
         print("  Twitter credentials found. Proceeding...")


    # Extract coin name from query for tagging (simple approach)
    # Handles cases like "$BTC" or "#bitcoin"
    query_parts = query.split()
    coin_name = query_parts[0].replace('$', '').replace('#', '').upper()

    client = Client('en-US')
    retries = 0
    delay = initial_delay
    tweet_list = [] # Initialize list outside the loop

    while retries < max_retries:
        try:
            # Log in (cookies are cached in cookies.json)
            # Ensure cookies_file path is correct if script is not in root dir
            # In Colab, it will be saved in the root content directory by default
            cookies_path = 'cookies.json'
            print(f"  Attempting Twitter login (attempt {retries + 1}) using cookies: {cookies_path}...")
            await client.login(
                auth_info_1=TWITTER_USERNAME,
                auth_info_2=TWITTER_EMAIL,
                password=TWITTER_PASSWORD,
                cookies_file=cookies_path
            )
            print("  Twitter login successful.")

            # Fetch tweets using search_tweet
            print(f"  Searching for tweets (max: {max_tweets})...")
            count = 0
            # --- FIX: Iterate over the result object's data ---
            search_result = await client.search_tweet(query, "Latest")

            # Check if search_result has tweets (adjust attribute name if needed, e.g., .data, .tweets)
            tweets_to_process = []
            if hasattr(search_result, 'data') and isinstance(search_result.data, list):
                 tweets_to_process = search_result.data
            elif isinstance(search_result, list): # If it directly returns a list
                 tweets_to_process = search_result
            else:
                 print(f"  Warning: Unexpected search result type: {type(search_result)}. Cannot process tweets.")

            for tweet in tweets_to_process:
                 if count >= max_tweets:
                     break

                 # Standardize timestamp
                 timestamp = standardize_timestamp(tweet.created_at)
                 if pd.isna(timestamp):
                      print(f"    Skipping tweet due to invalid timestamp: {tweet.created_at}")
                      continue # Skip tweets with unparseable timestamps

                 tweet_list.append({
                     "timestamp": timestamp,
                     "coin": coin_name,
                     "source": "Twitter",
                     "tweet_id": str(tweet.id), # Ensure tweet ID is string
                     "user": tweet.user.name if tweet.user else 'N/A', # Handle potential missing user
                     "tweet_text": tweet.text
                 })
                 count += 1
                 # Add a small delay within the loop to potentially avoid rapid-fire issues
                 # await asyncio.sleep(0.1) # Optional: uncomment if rate limits are still an issue

            print(f"  Fetched and processed {count} tweets.")
            # If successful, break out of the retry loop
            break
            # --- END FIX ---

        except TooManyRequests as e:
            retries += 1
            print(f"  Twitter rate limit hit (attempt {retries}/{max_retries}). Waiting {delay} seconds...")
            await asyncio.sleep(delay)
            delay *= 2 # Exponential backoff
        except Exception as e:
            print(f"  An unexpected error occurred during Twitter fetching: {e}")
            traceback.print_exc() # Print full traceback
            # Return empty DF immediately on unexpected errors
            return pd.DataFrame()

    if retries >= max_retries:
        print(f"  Failed to fetch Twitter data after {max_retries} retries due to rate limits.")
        return pd.DataFrame() # Return empty if max retries reached

    if not tweet_list:
         print(f"  No tweets processed for '{query}'.")
         return pd.DataFrame()

    df = pd.DataFrame(tweet_list)
    # Drop rows where timestamp is NaT after all attempts
    df = df.dropna(subset=['timestamp'])
    print(f"  Successfully processed {len(df)} tweets for '{query}'.")
    return df[['timestamp', 'coin', 'source', 'tweet_id', 'user', 'tweet_text']]


def fetch_twitter_data(query, max_tweets=100):
    """Synchronous wrapper for the async Twitter fetcher."""
    # Ensure an event loop exists or create one if running script directly
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
             print("Async event loop already running. Using nest_asyncio.")
             # nest_asyncio allows re-entering the loop
             df_result = loop.run_until_complete(fetch_twitter_data_async(query, max_tweets))
        else:
             df_result = loop.run_until_complete(fetch_twitter_data_async(query, max_tweets))
    except RuntimeError: # 'RuntimeError: There is no current event loop...'
        print("No current event loop found. Creating a new one.")
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        df_result = loop.run_until_complete(fetch_twitter_data_async(query, max_tweets))
        # loop.close() # Close the loop if we created it, optional

    return df_result


# --- Main Execution ---

if __name__ == "__main__":
    print("Starting data fetching process...")

    # --- Define Parameters ---
    COIN_ID = "BTC" # Focus on Bitcoin for this example
    PRICE_PRODUCT_ID = f"{COIN_ID}-USD"
    DAYS_HISTORY = 7 # Fetch last 7 days of data for quicker testing
    NEWS_QUERY = "Bitcoin"
    TWITTER_QUERY = f"{COIN_ID} lang:en" # Query for English tweets about the specific COIN_ID
    MAX_NEWS = 20 # Limit news articles
    MAX_TWEETS = 30 # Limit tweet fetching

    # --- Fetch Data ---
    df_prices = fetch_historical_prices(PRICE_PRODUCT_ID, days_history=DAYS_HISTORY)
    df_news = fetch_google_news_rss(NEWS_QUERY, max_articles=MAX_NEWS)
    df_tweets = fetch_twitter_data(TWITTER_QUERY, max_tweets=MAX_TWEETS)

    # --- Combine Data ---
    # This creates a long-format table where each row is one data point.
    all_data_list = []

    if df_prices is not None and not df_prices.empty:
        df_prices_melt = df_prices.copy()
        df_prices_melt['data_type'] = 'price_ohlcv'
        df_prices_melt['value'] = df_prices_melt.apply(lambda row: {
            'open': row['open'], 'high': row['high'], 'low': row['low'],
            'close': row['close'], 'volume': row['volume']
        }, axis=1)
        all_data_list.append(df_prices_melt[['timestamp', 'coin', 'source', 'data_type', 'value']])
        print(f"Price data added: {len(df_prices_melt)} rows")
    else:
        print("No price data fetched or DataFrame is empty.")


    if df_news is not None and not df_news.empty:
        df_news_melt = df_news.copy()
        df_news_melt['data_type'] = 'news'
        df_news_melt['value'] = df_news_melt.apply(lambda row: {
            'title': row['title'], 'url': row['url'] # 'content': row['content'] # Optionally include content
        }, axis=1)
        all_data_list.append(df_news_melt[['timestamp', 'coin', 'source', 'data_type', 'value']])
        print(f"News data added: {len(df_news_melt)} rows")
    else:
        print("No news data fetched or DataFrame is empty.")


    if df_tweets is not None and not df_tweets.empty:
        df_tweets_melt = df_tweets.copy()
        df_tweets_melt['data_type'] = 'tweet'
        df_tweets_melt['value'] = df_tweets_melt.apply(lambda row: {
            'tweet_id': row['tweet_id'], 'user': row['user'], 'text': row['tweet_text']
        }, axis=1)
        all_data_list.append(df_tweets_melt[['timestamp', 'coin', 'source', 'data_type', 'value']])
        print(f"Tweet data added: {len(df_tweets_melt)} rows")
    else:
        print("No tweet data fetched or DataFrame is empty.")


    if all_data_list:
        # Concatenate all dataframes in the list
        df_combined_raw = pd.concat(all_data_list, ignore_index=True)
        # Sort the final dataframe by timestamp (most recent first)
        df_combined_raw = df_combined_raw.sort_values('timestamp', ascending=False)

        # --- Save Combined Raw Data ---
        output_filename = "combined_crypto_data_raw.csv"
        try:
            df_combined_raw.to_csv(output_filename, index=False, encoding='utf-8')
            print(f"\nCombined raw data saved to {output_filename}")
            print(f"Total rows combined: {len(df_combined_raw)}")

            print("\nSample of combined data (most recent entries):")
            # Display more columns for better context
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', 1000)
            print(df_combined_raw.head())
        except Exception as e:
            print(f"\nError saving combined data to CSV: {e}")
            traceback.print_exc() # Print traceback on save error

    else:
        print("\nNo data fetched from any source.")

    print("\nData fetching process finished.")


Starting data fetching process...
Fetching historical prices for BTC-USD...
  Fetching data from 2025-04-23T04:39:06 to 2025-04-30T04:39:06
  Successfully fetched 7 price data points for BTC-USD.
Fetching Google News RSS for query: 'Bitcoin'...
  Found 103 potential news items.
  Successfully processed 20 news items for 'Bitcoin'.
Async event loop already running. Using nest_asyncio.
Fetching Twitter data for query: 'BTC lang:en'...
  Twitter credentials found. Proceeding...
  Attempting Twitter login (attempt 1) using cookies: cookies.json...
  Twitter login successful.
  Searching for tweets (max: 30)...
  An unexpected error occurred during Twitter fetching: status: 404, message: ""
Price data added: 7 rows
News data added: 20 rows
No tweet data fetched or DataFrame is empty.

Combined raw data saved to combined_crypto_data_raw.csv
Total rows combined: 27

Sample of combined data (most recent entries):
                   timestamp     coin      source    data_type                   

Traceback (most recent call last):
  File "<ipython-input-3-a7a76c579901>", line 361, in fetch_twitter_data_async
    search_result = await client.search_tweet(query, "Latest")
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/twikit/client/client.py", line 731, in search_tweet
    response, _ = await self.gql.search_timeline(query, product, count, cursor)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/twikit/client/gql.py", line 159, in search_timeline
    return await self.gql_get(Endpoint.SEARCH_TIMELINE, variables, FEATURES)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/twikit/client/gql.py", line 124, in gql_get
    return await self.base.get(url, params=flatten_params(params), headers=headers, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^